# 🛰️ Segmentación de Imágenes Satelitales con K-Means

**Objetivo:** Clasificar coberturas del suelo (bosque, agua, urbano) a partir de una imagen satelital usando segmentación por color con K-Means.

**Librerías requeridas:** `opencv-python`, `numpy`, `matplotlib`, `scikit-learn`, `Pillow`, `ipywidgets`

---
### Instalación de dependencias

In [ ]:
# Instalar librerías si no están disponibles
import subprocess, sys

packages = ['opencv-python', 'numpy', 'matplotlib', 'scikit-learn', 'Pillow', 'ipywidgets']
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('✅ Dependencias instaladas correctamente.')

---
## 📦 Celda 1 — Importaciones y configuración global

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from sklearn.cluster import KMeans
from PIL import Image
import warnings
import os

warnings.filterwarnings('ignore')

# --- Paleta de colores para las clases (RGB normalizado) ---
# Ajusta los colores y etiquetas según tu imagen
PALETA = {
    0: {'color': (0.20, 0.55, 0.20), 'nombre': 'Bosque/Vegetación'},
    1: {'color': (0.18, 0.40, 0.75), 'nombre': 'Agua'},
    2: {'color': (0.72, 0.72, 0.72), 'nombre': 'Urbano/Suelo'},
    3: {'color': (0.85, 0.75, 0.45), 'nombre': 'Arena/Cultivos'},
    4: {'color': (0.60, 0.20, 0.20), 'nombre': 'Zona quemada'},
}

print('✅ Librerías cargadas.')

---
## 🖼️ Celda 2 — Cargar imagen satelital

Puedes usar:
- Una imagen propia (`.jpg`, `.png`, `.tif`)
- La imagen sintética generada en esta celda como demo

In [ ]:
# ============================================================
# OPCIÓN A: Cargar tu propia imagen
# Descomenta la línea siguiente y escribe la ruta de tu imagen
# IMAGE_PATH = 'imagen_satelital.jpg'
# ============================================================

# ============================================================
# OPCIÓN B: Generar imagen sintética de demostración
# ============================================================
def generar_imagen_demo(alto=512, ancho=512, seed=42):
    """Genera una imagen RGB sintética con zonas de agua, bosque y urbano."""
    rng = np.random.default_rng(seed)
    img = np.zeros((alto, ancho, 3), dtype=np.uint8)

    # Zona de agua (azul oscuro)
    img[0:180, 0:ancho] = [35, 80, 160]
    img[0:180, 0:ancho] += rng.integers(-15, 15, (180, ancho, 3), dtype=np.int8).view(np.uint8)

    # Zona de bosque (verde oscuro)
    img[180:370, 0:300] = [34, 110, 40]
    img[180:370, 0:300] += rng.integers(-20, 20, (190, 300, 3), dtype=np.int8).view(np.uint8)

    # Zona urbana (gris)
    img[180:370, 300:ancho] = [160, 158, 155]
    img[180:370, 300:ancho] += rng.integers(-18, 18, (190, 212, 3), dtype=np.int8).view(np.uint8)

    # Zona de cultivos / arena (amarillo)
    img[370:alto, 0:ancho] = [195, 175, 95]
    img[370:alto, 0:ancho] += rng.integers(-20, 20, (alto-370, ancho, 3), dtype=np.int8).view(np.uint8)

    img = np.clip(img, 0, 255).astype(np.uint8)
    # Suavizar bordes para mayor realismo
    img = cv2.GaussianBlur(img, (7, 7), 0)
    return img


IMAGE_PATH = '../media/San francisco.jpg'  # Cambia a tu ruta para usar imagen real

if IMAGE_PATH and os.path.exists(IMAGE_PATH):
    image_bgr = cv2.imread(IMAGE_PATH)
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    print(f'✅ Imagen cargada desde: {IMAGE_PATH}')
else:
    image_rgb = generar_imagen_demo()
    print('⚠️  Usando imagen sintética de demostración.')

print(f'📐 Tamaño de imagen: {image_rgb.shape[1]}×{image_rgb.shape[0]} px')

plt.figure(figsize=(8, 6))
plt.imshow(image_rgb)
plt.title('Imagen Satelital Original', fontsize=13)
plt.axis('off')
plt.tight_layout()
plt.show()

---
## ✂️ Celda 3 — Seleccionar Región de Interés (ROI)

**Dos modos disponibles:**
- `modo = 'manual'` → definir coordenadas en el código
- `modo = 'interactivo'` → usar `cv2.selectROI()` (requiere entorno de escritorio)
- `modo = 'completa'` → usar toda la imagen

In [ ]:
# ── Configuración de ROI ──────────────────────────────────────
modo = 'completa'          # 'manual' | 'interactivo' | 'completa'

# Coordenadas para modo manual: (x_inicio, y_inicio, ancho, alto)
ROI_MANUAL = (50, 50, 420, 410)
# ─────────────────────────────────────────────────────────────

def seleccionar_roi(imagen, modo, roi_manual):
    h, w = imagen.shape[:2]

    if modo == 'interactivo':
        print('🖱️  Dibuja el ROI en la ventana emergente y presiona ENTER o ESPACIO.')
        img_bgr = cv2.cvtColor(imagen, cv2.COLOR_RGB2BGR)
        r = cv2.selectROI('Selecciona ROI (ENTER para confirmar)', img_bgr)
        cv2.destroyAllWindows()
        if r[2] == 0 or r[3] == 0:
            print('⚠️  ROI vacío. Usando imagen completa.')
            return imagen, (0, 0, w, h)
        roi = imagen[int(r[1]):int(r[1]+r[3]), int(r[0]):int(r[0]+r[2])]
        return roi, r

    elif modo == 'manual':
        x, y, rw, rh = roi_manual
        # Clamp a límites de imagen
        x, y = max(0, x), max(0, y)
        rw, rh = min(rw, w - x), min(rh, h - y)
        roi = imagen[y:y+rh, x:x+rw]
        return roi, (x, y, rw, rh)

    else:  # 'completa'
        return imagen.copy(), (0, 0, w, h)


roi, roi_coords = seleccionar_roi(image_rgb, modo, ROI_MANUAL)
x0, y0, rw, rh = roi_coords

# Visualizar ROI sobre imagen original
imagen_marcada = image_rgb.copy()
cv2.rectangle(imagen_marcada, (x0, y0), (x0+rw, y0+rh), (255, 50, 50), 3)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].imshow(imagen_marcada)
axes[0].set_title('Imagen completa + ROI (rojo)', fontsize=12)
axes[0].axis('off')
axes[1].imshow(roi)
axes[1].set_title(f'ROI seleccionada  ({rw}×{rh} px)', fontsize=12)
axes[1].axis('off')
plt.tight_layout()
plt.show()
print(f'📐 ROI: x={x0}, y={y0}, ancho={rw}, alto={rh}')

---
## 🤖 Celda 4 — Clasificación por color con K-Means

In [ ]:
def aplicar_kmeans(imagen_roi, n_clusters=3, random_state=42):
    """Aplica K-Means sobre los píxeles RGB de la ROI."""
    h, w = imagen_roi.shape[:2]
    pixels = imagen_roi.reshape(-1, 3).astype(np.float32)

    kmeans = KMeans(
        n_clusters=n_clusters,
        random_state=random_state,
        n_init='auto',
        max_iter=300
    )
    kmeans.fit(pixels)

    etiquetas = kmeans.labels_.reshape(h, w)
    centros = kmeans.cluster_centers_.astype(np.uint8)  # color medio por cluster
    inercia = kmeans.inertia_

    # ---- NUEVO: Asignación dinámica de PALETA inteligente ----
    global PALETA
    PALETA = {}
    
    referencias = [
        ({'color': (0.18, 0.40, 0.75), 'nombre': 'Agua'}, np.array([40, 60, 90])),
        ({'color': (0.20, 0.55, 0.20), 'nombre': 'Bosque/Vegetación'}, np.array([50, 80, 50])),
        ({'color': (0.72, 0.72, 0.72), 'nombre': 'Urbano/Suelo'}, np.array([150, 150, 150])),
        ({'color': (0.85, 0.75, 0.45), 'nombre': 'Arena/Cultivos'}, np.array([160, 140, 100])),
        ({'color': (0.60, 0.20, 0.20), 'nombre': 'Zona quemada'}, np.array([120, 60, 60]))
    ]
    
    for i, centro in enumerate(centros):
        mejor_ref = min(referencias, key=lambda r: np.linalg.norm(centro - r[1]))[0]
        PALETA[i] = mejor_ref

    return etiquetas, centros, inercia


# ── Número de clases ──────────────────────────────────────────
N_CLUSTERS = 4   # Modifica según tu imagen (2-6 recomendado)
# ─────────────────────────────────────────────────────────────

print(f'⏳ Ejecutando K-Means con {N_CLUSTERS} clusters...')
etiquetas, centros, inercia = aplicar_kmeans(roi, n_clusters=N_CLUSTERS)
print(f'✅ K-Means completado.  Inercia: {inercia:,.0f}')

# Mostrar colores medios por cluster
fig, axes = plt.subplots(1, N_CLUSTERS, figsize=(N_CLUSTERS * 2.2, 2))
for i, (ax, color) in enumerate(zip(axes, centros)):
    muestra = np.full((60, 100, 3), color, dtype=np.uint8)
    ax.imshow(muestra)
    ax.set_title(f'Clase {i}\nRGB{tuple(color)}', fontsize=8)
    ax.axis('off')
plt.suptitle('Centros de cluster (color medio por clase)', fontsize=11, y=1.05)
plt.tight_layout()
plt.show()

---
## 🎨 Celda 5 — Visualización de resultados

Se colorean las clases, se dibujan contornos y se etiquetan zonas.

In [ ]:
def construir_imagen_segmentada(etiquetas, n_clusters, paleta):
    """Colorea cada clase según la paleta definida."""
    h, w = etiquetas.shape
    img_color = np.zeros((h, w, 3), dtype=np.float32)
    for k in range(n_clusters):
        c = paleta.get(k, {'color': (0.5, 0.5, 0.5)})
        mask = etiquetas == k
        img_color[mask] = c['color']
    return img_color


def dibujar_contornos(etiquetas, imagen_color, n_clusters):
    """Dibuja contornos de cada clase sobre la imagen coloreada."""
    img_out = (imagen_color * 255).astype(np.uint8).copy()
    for k in range(n_clusters):
        mask = (etiquetas == k).astype(np.uint8) * 255
        contornos, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        # Filtrar contornos muy pequeños (ruido)
        contornos = [c for c in contornos if cv2.contourArea(c) > 300]
        cv2.drawContours(img_out, contornos, -1, (255, 255, 255), 1)
    return img_out


def calcular_porcentajes(etiquetas, n_clusters):
    total = etiquetas.size
    return {k: round(np.sum(etiquetas == k) / total * 100, 2) for k in range(n_clusters)}


# Construir visualización
img_seg_color = construir_imagen_segmentada(etiquetas, N_CLUSTERS, PALETA)
img_seg_contornos = dibujar_contornos(etiquetas, img_seg_color, N_CLUSTERS)
porcentajes = calcular_porcentajes(etiquetas, N_CLUSTERS)

# ── Leyenda dinámica ──────────────────────────────────────────
parches = []
for k in range(N_CLUSTERS):
    c = PALETA.get(k, {'color': (0.5, 0.5, 0.5), 'nombre': f'Clase {k}'})
    pct = porcentajes[k]
    parches.append(mpatches.Patch(color=c['color'], label=f"{c['nombre']} ({pct}%)"))

# ── Figura comparativa ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(roi)
axes[0].set_title('ROI Original', fontsize=12)
axes[0].axis('off')

axes[1].imshow(img_seg_color)
axes[1].set_title(f'Segmentación K-Means  (k={N_CLUSTERS})', fontsize=12)
axes[1].axis('off')
axes[1].legend(handles=parches, loc='lower left', fontsize=8, framealpha=0.85)

axes[2].imshow(img_seg_contornos)
axes[2].set_title('Segmentación + Contornos', fontsize=12)
axes[2].axis('off')

plt.suptitle('Clasificación de coberturas del suelo por K-Means', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Tabla de porcentajes
print('\n📊 Distribución de clases:')
print(f'{"Clase":<6} {"Nombre":<25} {"Porcentaje":>12}')
print('-' * 45)
for k in range(N_CLUSTERS):
    nombre = PALETA.get(k, {}).get('nombre', f'Clase {k}')
    print(f'{k:<6} {nombre:<25} {porcentajes[k]:>10.2f}%')

---
## 💾 Celda 6 — Guardar máscaras binarias por clase

In [ ]:
import os

OUTPUT_DIR = 'mascaras_salida'
os.makedirs(OUTPUT_DIR, exist_ok=True)

nombres_archivo = ['bosque', 'agua', 'urbano', 'cultivos', 'quemado']

guardados = []
for k in range(N_CLUSTERS):
    mascara = (etiquetas == k).astype(np.uint8) * 255
    nombre_clase = nombres_archivo[k] if k < len(nombres_archivo) else f'clase_{k}'
    ruta = os.path.join(OUTPUT_DIR, f'{nombre_clase}.png')
    cv2.imwrite(ruta, mascara)
    guardados.append(ruta)
    print(f'💾 Guardado: {ruta}  ({np.sum(mascara > 0):,} píxeles)')

# Guardar imagen segmentada con colores
ruta_seg = os.path.join(OUTPUT_DIR, 'segmentacion_color.png')
seg_bgr = cv2.cvtColor((img_seg_color * 255).astype(np.uint8), cv2.COLOR_RGB2BGR)
cv2.imwrite(ruta_seg, seg_bgr)
print(f'\n🖼️  Segmentación coloreada: {ruta_seg}')

# Previsualizar máscaras
fig, axes = plt.subplots(1, N_CLUSTERS, figsize=(N_CLUSTERS * 3, 3))
for k, (ax, ruta) in enumerate(zip(axes, guardados)):
    mascara = cv2.imread(ruta, cv2.IMREAD_GRAYSCALE)
    nombre_clase = PALETA.get(k, {}).get('nombre', f'Clase {k}')
    ax.imshow(mascara, cmap='gray')
    ax.set_title(nombre_clase, fontsize=9)
    ax.axis('off')
plt.suptitle('Máscaras binarias exportadas', fontsize=12)
plt.tight_layout()
plt.show()

---
## ⚖️ Celda 7 — BONUS: K-Means vs Umbral de Color (Color Thresholding)

In [ ]:
def segmentar_por_umbral(imagen_rgb):
    """
    Segmenta por rangos de color en espacio HSV.
    Retorna máscara con etiquetas: 0=vegetación, 1=agua, 2=otro.
    """
    hsv = cv2.cvtColor(imagen_rgb, cv2.COLOR_RGB2HSV)
    resultado = np.full(imagen_rgb.shape[:2], 2, dtype=np.uint8)  # defecto: 'otro'

    # Vegetación (verde)
    mask_veg1 = cv2.inRange(hsv, np.array([35, 40, 40]),  np.array([90, 255, 255]))
    resultado[mask_veg1 > 0] = 0

    # Agua (azul)
    mask_agua = cv2.inRange(hsv, np.array([100, 50, 50]), np.array([140, 255, 255]))
    resultado[mask_agua > 0] = 1

    return resultado


etiquetas_umbral = segmentar_por_umbral(roi)

paleta_umbral = {
    0: {'color': (0.20, 0.55, 0.20), 'nombre': 'Vegetación (umbral)'},
    1: {'color': (0.18, 0.40, 0.75), 'nombre': 'Agua (umbral)'},
    2: {'color': (0.70, 0.68, 0.65), 'nombre': 'Otro (umbral)'},
}
img_umbral_color = construir_imagen_segmentada(etiquetas_umbral, 3, paleta_umbral)

# Métricas de cobertura
pct_kmeans  = calcular_porcentajes(etiquetas, N_CLUSTERS)
pct_umbral  = calcular_porcentajes(etiquetas_umbral, 3)

# ── Figura comparativa ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(roi)
axes[0].set_title('ROI Original', fontsize=12)
axes[0].axis('off')

parches_km = [mpatches.Patch(
    color=PALETA.get(k, {'color': (0.5,0.5,0.5)})['color'],
    label=f"{PALETA.get(k, {'nombre': f'Clase {k}'})['nombre']} ({pct_kmeans[k]}%)"
) for k in range(N_CLUSTERS)]
axes[1].imshow(img_seg_color)
axes[1].set_title(f'K-Means  (k={N_CLUSTERS})', fontsize=12)
axes[1].axis('off')
axes[1].legend(handles=parches_km, loc='lower left', fontsize=7, framealpha=0.85)

parches_umb = [mpatches.Patch(
    color=paleta_umbral[k]['color'],
    label=f"{paleta_umbral[k]['nombre']} ({pct_umbral[k]}%)"
) for k in range(3)]
axes[2].imshow(img_umbral_color)
axes[2].set_title('Umbral de Color (HSV)', fontsize=12)
axes[2].axis('off')
axes[2].legend(handles=parches_umb, loc='lower left', fontsize=7, framealpha=0.85)

plt.suptitle('Comparación: K-Means vs Umbral de Color', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('\n📋 Comparación de métodos:')
print(f'{"Método":<30} {"Ventajas":<40} {"Desventajas"}')
print('-' * 90)
print(f'{"K-Means":<30} {"Aprende colores automáticamente":<40} Necesita definir k; no supervizado')
print(f'{"Umbral HSV":<30} {"Rápido; interpretable":<40} Requiere ajuste manual por imagen')

---
## 📊 Celda 8 — Análisis del número óptimo de clusters (Elbow Method)

In [ ]:
def metodo_codo(imagen_roi, k_min=2, k_max=8, muestra=5000, seed=42):
    """Calcula la inercia para distintos valores de k (método del codo)."""
    pixels = imagen_roi.reshape(-1, 3).astype(np.float32)

    # Sub-muestreo para velocidad
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(pixels), size=min(muestra, len(pixels)), replace=False)
    pixels_muestra = pixels[idx]

    inercias = []
    rango_k = range(k_min, k_max + 1)
    for k in rango_k:
        km = KMeans(n_clusters=k, random_state=seed, n_init='auto', max_iter=100)
        km.fit(pixels_muestra)
        inercias.append(km.inertia_)
        print(f'  k={k}  →  inercia={km.inertia_:,.0f}')

    return list(rango_k), inercias


print('⏳ Calculando método del codo...')
ks, inercias = metodo_codo(roi)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ks, inercias, 'o-', color='#2c6fad', linewidth=2, markersize=7)
ax.axvline(x=N_CLUSTERS, color='crimson', linestyle='--', alpha=0.7, label=f'k actual = {N_CLUSTERS}')
ax.set_xlabel('Número de clusters (k)', fontsize=12)
ax.set_ylabel('Inercia (WCSS)', fontsize=12)
ax.set_title('Método del Codo — Elección de k óptimo', fontsize=13)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print('💡 Elige k en el punto donde la curva comienza a aplanarse ("codo").')

---
## 🎛️ Celda 9 — Interfaz Interactiva con `ipywidgets`

> **Requiere:** `ipywidgets` instalado y extensiones de Jupyter habilitadas.
> Ejecuta `jupyter nbextension enable --py widgetsnbextension` si los widgets no se ven.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    WIDGETS_OK = True
except ImportError:
    WIDGETS_OK = False
    print('⚠️  ipywidgets no disponible. Instala con: pip install ipywidgets')

if WIDGETS_OK:
    # ── Controles ─────────────────────────────────────────────
    slider_k = widgets.IntSlider(
        value=N_CLUSTERS, min=2, max=8, step=1,
        description='Clusters k:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='400px')
    )
    boton_exportar = widgets.Button(
        description='💾 Exportar segmentación',
        button_style='success',
        layout=widgets.Layout(width='220px')
    )
    salida = widgets.Output()

    def actualizar_segmentacion(change):
        k_val = slider_k.value
        with salida:
            clear_output(wait=True)
            etas, cts, _ = aplicar_kmeans(roi, n_clusters=k_val)
            img_c = construir_imagen_segmentada(etas, k_val, PALETA)
            img_cnt = dibujar_contornos(etas, img_c, k_val)
            pcts = calcular_porcentajes(etas, k_val)

            fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
            axes[0].imshow(roi);             axes[0].set_title('Original'); axes[0].axis('off')
            axes[1].imshow(img_c);           axes[1].set_title(f'K-Means k={k_val}'); axes[1].axis('off')
            axes[2].imshow(img_cnt);         axes[2].set_title('+ Contornos'); axes[2].axis('off')

            parches_w = [mpatches.Patch(
                color=PALETA.get(i, {'color': (0.5,0.5,0.5)})['color'],
                label=f"{PALETA.get(i, {'nombre': f'Clase {i}'})['nombre']} ({pcts[i]}%)"
            ) for i in range(k_val)]
            axes[1].legend(handles=parches_w, loc='lower left', fontsize=7, framealpha=0.85)
            plt.tight_layout()
            plt.show()

    def exportar_resultado(_):
        k_val = slider_k.value
        etas, _, _ = aplicar_kmeans(roi, n_clusters=k_val)
        img_c = construir_imagen_segmentada(etas, k_val, PALETA)
        ruta = os.path.join(OUTPUT_DIR, f'segmentacion_k{k_val}_widget.png')
        seg_bgr = cv2.cvtColor((img_c * 255).astype(np.uint8), cv2.COLOR_RGB2BGR)
        cv2.imwrite(ruta, seg_bgr)
        with salida:
            print(f'\n✅ Exportado: {ruta}')

    slider_k.observe(actualizar_segmentacion, names='value')
    boton_exportar.on_click(exportar_resultado)

    display(widgets.VBox([
        widgets.HBox([slider_k, boton_exportar]),
        salida
    ]))
    actualizar_segmentacion(None)  # render inicial

else:
    print('Interfaz interactiva no disponible. Usa la Celda 5 para visualizar resultados.')

---
## 📝 Resumen del Flujo

```
Imagen Satelital
      │
      ▼
  Cargar (cv2.imread)
      │
      ▼
  Seleccionar ROI  ──── manual / interactivo / completa
      │
      ▼
  K-Means (sklearn)  ──── elige k con método del codo
      │
      ├──► Imagen segmentada coloreada
      ├──► Contornos (cv2.findContours)
      ├──► Máscaras binarias por clase (.png)
      └──► Comparación con umbral HSV
```

### 🔗 Referencias
- [OpenCV Docs](https://docs.opencv.org/)  
- [scikit-learn KMeans](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html)  
- [Segmentación de imágenes satelitales — ESA](https://www.esa.int/Applications/Observing_the_Earth)